In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)


from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import joblib

train_df = joblib.load('drive/MyDrive/nlp/train_df.pkl')
val_df = joblib.load('drive/MyDrive/nlp/val_df.pkl')
test_df = joblib.load('drive/MyDrive/nlp/test_df.pkl')

In [ ]:
train_df.head()

,text,variety,source,Sentiment,Sarcasm,tokens,clean_text
0,I'm a member of the Green Party but I'll be vo...,en-UK,Reddit,0.0,0.0,"[member, but, vote, lib, dem, so, tight, tory,...",member but vote lib dem so tight tory not cont...
1,Yeah it blew out to 3x what it was budgeted fo...,en-AU,Reddit,0.0,1.0,"[yeah, blow, 3x, what, budget, who, would, ve,...",yeah blow 3x what budget who would ve think gi...
2,"Food was pretty great. A little dry, but I am ...",en-AU,Google,1.0,0.0,"[food, pretty, great, little, dry, but, sucker...",food pretty great little dry but sucker load d...
3,Firstly the staff seemed as if they did n't wa...,en-UK,Google,0.0,0.0,"[firstly, staff, they, want, cheese, ham, toas...",firstly staff they want cheese ham toasty orde...
4,We came for lunch and enjoyed the food we orde...,en-UK,Google,1.0,0.0,"[we, come, lunch, enjoy, food, we, order, hot,...",we come lunch enjoy food we order hot taste fr...


### Compute metrics for baseline

In [ ]:
def compute_metrics_from_preds(y_true, y_pred):
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    cls_report = classification_report(y_true, y_pred, output_dict=True)

    return {
        "accuracy": acc,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_f1": f1_weighted,
        "classification_report": cls_report
    }


## Basic Baseline setup

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))
X_train = vectorizer.fit_transform(train_df['clean_text'])
X_test = vectorizer.transform(test_df['clean_text'])

y_train = train_df['Sentiment']
y_test = test_df['Sentiment']

y_train_s = train_df['Sarcasm']
y_test_s = test_df['Sarcasm']

In [ ]:
X_train.shape, X_test.shape

((3747, 5000), (2183, 5000))

In [ ]:
y_train.value_counts(), y_test.value_counts()

(Sentiment
 0.0    1907
 1.0    1840
 Name: count, dtype: int64,
 Sentiment
 0.0    1117
 1.0    1066
 Name: count, dtype: int64)

In [ ]:
y_train_s.value_counts(), y_test_s.value_counts()

(Sarcasm
 0.0    3223
 1.0     524
 Name: count, dtype: int64,
 Sarcasm
 0.0    1878
 1.0     305
 Name: count, dtype: int64)

In [ ]:
model = LogisticRegression(random_state=42, solver='liblinear', max_iter=500, C=1.0, class_weight='balanced')

### Sentiment

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
compute_metrics_from_preds(y_test, y_pred)

{'accuracy': 0.8314246449839671,
 'macro_precision': 0.8339276861749086,
 'macro_recall': 0.830368885432536,
 'macro_f1': 0.8307184242777743,
 'weighted_f1': 0.8309738658098015,
 'classification_report': {'0.0': {'precision': 0.8102734051367025,
   'recall': 0.8755595344673232,
   'f1-score': 0.8416523235800344,
   'support': 1117.0},
  '1.0': {'precision': 0.8575819672131147,
   'recall': 0.7851782363977486,
   'f1-score': 0.8197845249755142,
   'support': 1066.0},
  'accuracy': 0.8314246449839671,
  'macro avg': {'precision': 0.8339276861749086,
   'recall': 0.830368885432536,
   'f1-score': 0.8307184242777743,
   'support': 2183.0},
  'weighted avg': {'precision': 0.8333750666911943,
   'recall': 0.8314246449839671,
   'f1-score': 0.8309738658098015,
   'support': 2183.0}}}

### Sarcasm

In [ ]:
model.fit(X_train, y_train_s)
y_pred_s = model.predict(X_test)

In [ ]:
compute_metrics_from_preds(y_test_s, y_pred_s)

{'accuracy': 0.7370590929912963,
 'macro_precision': 0.6100856779669499,
 'macro_recall': 0.692017144154053,
 'macro_f1': 0.6162039215109726,
 'weighted_f1': 0.7713918274199925,
 'classification_report': {'0.0': {'precision': 0.9261437908496732,
   'recall': 0.7545260915867945,
   'f1-score': 0.8315727699530516,
   'support': 1878.0},
  '1.0': {'precision': 0.29402756508422667,
   'recall': 0.6295081967213115,
   'f1-score': 0.40083507306889354,
   'support': 305.0},
  'accuracy': 0.7370590929912963,
  'macro avg': {'precision': 0.6100856779669499,
   'recall': 0.692017144154053,
   'f1-score': 0.6162039215109726,
   'support': 2183.0},
  'weighted avg': {'precision': 0.8378270483583947,
   'recall': 0.7370590929912963,
   'f1-score': 0.7713918274199925,
   'support': 2183.0}}}

## Tuning for TF-IDF+LR

In [ ]:
def find_best_config(label_col, candidate_configs):

    train_local = train_df[["clean_text", label_col]].copy()
    val_local = val_df[["clean_text", label_col]].copy()
    test_local = test_df[["clean_text", label_col]].copy()

    best_config = None
    best_val_f1 = -1
    tuning_rows = []

    # Tune on validation
    for config in candidate_configs:
        vectorizer = TfidfVectorizer(
            max_features=config["max_features"],
            ngram_range=config["ngram_range"]
        )

        X_train = vectorizer.fit_transform(train_local["clean_text"])
        X_val = vectorizer.transform(val_local["clean_text"])

        y_train = train_local[label_col]
        y_val = val_local[label_col]

        clf = LogisticRegression(
            max_iter=config["max_iter"],
            solver="liblinear",
            C=config["C"],
            class_weight=config["class_weight"],
            random_state=42
        )
        clf.fit(X_train, y_train)
        val_pred = clf.predict(X_val)
        val_metrics = compute_metrics_from_preds(y_val, val_pred)

        row = config.copy()
        row["val_macro_f1"] = val_metrics["macro_f1"]
        tuning_rows.append(row)

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_config = config

    tuning_df = pd.DataFrame(tuning_rows).sort_values("val_macro_f1", ascending=False)
    print(f"\nBest baseline config for {label_col}:")
    print(best_config)
    display(tuning_df)
    return best_config

In [ ]:
candidate_configs = [
      {"max_features": 5000, "ngram_range": (1, 1), "C": 1.0, "class_weight": None, "max_iter": 500},
      {"max_features": 5000, "ngram_range": (1, 2), "C": 1.0, "class_weight": None, "max_iter": 500},
      {"max_features": 5000, "ngram_range": (1, 2), "C": 1.0, "class_weight": "balanced", "max_iter": 500},
      {"max_features": 10000, "ngram_range": (1, 2), "C": 1.0, "class_weight": "balanced", "max_iter": 500},
      {"max_features": 10000, "ngram_range": (1, 2), "C": 2.0, "class_weight": "balanced", "max_iter": 500},
      {"max_features": 5000, "ngram_range": (1, 1), "C": 1.0, "class_weight": None, "max_iter": 1000},
      {"max_features": 5000, "ngram_range": (1, 2), "C": 1.0, "class_weight": None, "max_iter": 1000},
      {"max_features": 5000, "ngram_range": (1, 2), "C": 1.0, "class_weight": "balanced", "max_iter": 1000},
      {"max_features": 10000, "ngram_range": (1, 2), "C": 1.0, "class_weight": "balanced", "max_iter": 1000},
      {"max_features": 10000, "ngram_range": (1, 2), "C": 2.0, "class_weight": "balanced", "max_iter": 1000}
  ]

best_sentiment = find_best_config("Sentiment", candidate_configs)
best_sarcasm = find_best_config("Sarcasm", candidate_configs)


Best baseline config for Sentiment:
{'max_features': 10000, 'ngram_range': (1, 2), 'C': 2.0, 'class_weight': 'balanced', 'max_iter': 500}


,max_features,ngram_range,C,class_weight,max_iter,val_macro_f1
9,10000,"(1, 2)",2.0,balanced,1000,0.837061
4,10000,"(1, 2)",2.0,balanced,500,0.837061
1,5000,"(1, 2)",1.0,None,500,0.837001
0,5000,"(1, 1)",1.0,None,500,0.837001
6,5000,"(1, 2)",1.0,None,1000,0.837001
5,5000,"(1, 1)",1.0,None,1000,0.837001
2,5000,"(1, 2)",1.0,balanced,500,0.833864
3,10000,"(1, 2)",1.0,balanced,500,0.833864
7,5000,"(1, 2)",1.0,balanced,1000,0.833864
8,10000,"(1, 2)",1.0,balanced,1000,0.833864



Best baseline config for Sarcasm:
{'max_features': 10000, 'ngram_range': (1, 2), 'C': 2.0, 'class_weight': 'balanced', 'max_iter': 500}


,max_features,ngram_range,C,class_weight,max_iter,val_macro_f1
9,10000,"(1, 2)",2.0,balanced,1000,0.662903
4,10000,"(1, 2)",2.0,balanced,500,0.662903
8,10000,"(1, 2)",1.0,balanced,1000,0.657290
3,10000,"(1, 2)",1.0,balanced,500,0.657290
7,5000,"(1, 2)",1.0,balanced,1000,0.655292
2,5000,"(1, 2)",1.0,balanced,500,0.655292
5,5000,"(1, 1)",1.0,None,1000,0.462199
0,5000,"(1, 1)",1.0,None,500,0.462199
1,5000,"(1, 2)",1.0,None,500,0.461274
6,5000,"(1, 2)",1.0,None,1000,0.461274


## Running the model with best hyperparameters

In [ ]:
sentiment_vectorizer = TfidfVectorizer(
      max_features=best_sentiment["max_features"],
      ngram_range=best_sentiment["ngram_range"]
  )

X_train = sentiment_vectorizer.fit_transform(train_df["clean_text"])
X_test = sentiment_vectorizer.transform(test_df["clean_text"])

y_train = train_df["Sentiment"]
y_test = test_df["Sentiment"]

sarcasm_vectorizer = TfidfVectorizer(
      max_features=best_sarcasm["max_features"],
      ngram_range=best_sarcasm["ngram_range"]
  )

X_train_s = sarcasm_vectorizer.fit_transform(train_df["clean_text"])
X_test_s = sarcasm_vectorizer.transform(test_df["clean_text"])

y_train_s = train_df["Sarcasm"]
y_test_s = test_df["Sarcasm"]

In [ ]:
X_train.shape, X_train_s.shape

((3747, 10000), (3747, 10000))

In [ ]:
sentiment_results = {}
sarcasm_results = {}

seeds = [42,100]

for seed in seeds:

  sentiment_model = LogisticRegression(
      max_iter=best_sentiment["max_iter"],
      solver="liblinear",
      C=best_sentiment["C"],
      class_weight=best_sentiment["class_weight"],
      random_state=seed
  )

  sarcasm_model = LogisticRegression(
      max_iter=best_sarcasm["max_iter"],
      solver="liblinear",
      C=best_sarcasm["C"],
      class_weight=best_sarcasm["class_weight"],
      random_state=seed
  )

  sentiment_model.fit(X_train, y_train)
  sarcasm_model.fit(X_train_s, y_train_s)

  y_pred = sentiment_model.predict(X_test)
  y_pred_s = sarcasm_model.predict(X_test_s)

  sentiment_metrics = compute_metrics_from_preds(y_test, y_pred)
  sarcasm_metrics = compute_metrics_from_preds(y_test_s, y_pred_s)

  sentiment_results[seed] = sentiment_metrics
  sarcasm_results[seed] = sarcasm_metrics
  print(f"Seed {seed} done")


Seed 42 done
Seed 100 done


In [ ]:
sentiment_results_pd = pd.DataFrame(sentiment_results)
sentiment_results_pd

,42,100
accuracy,0.830508,0.830508
macro_precision,0.831628,0.831628
macro_recall,0.829773,0.829773
macro_f1,0.830069,0.830069
weighted_f1,0.830271,0.830271
classification_report,"{'0.0': {'precision': 0.8173322005097706, 'rec...","{'0.0': {'precision': 0.8173322005097706, 'rec..."


In [ ]:
sarcasm_results_pd = pd.DataFrame(sarcasm_results)
sarcasm_results_pd

,42,100
accuracy,0.776454,0.776454
macro_precision,0.606493,0.606493
macro_recall,0.650378,0.650378
macro_f1,0.618371,0.618371
weighted_f1,0.795357,0.795357
classification_report,"{'0.0': {'precision': 0.9064327485380117, 'rec...","{'0.0': {'precision': 0.9064327485380117, 'rec..."


# Transformer

In [ ]:
!pip install transformers datasets evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


In [ ]:
train_df.head()

,text,variety,source,Sentiment,Sarcasm,tokens,clean_text
0,I'm a member of the Green Party but I'll be vo...,en-UK,Reddit,0.0,0.0,"[member, but, vote, lib, dem, so, tight, tory,...",member but vote lib dem so tight tory not cont...
1,Yeah it blew out to 3x what it was budgeted fo...,en-AU,Reddit,0.0,1.0,"[yeah, blow, 3x, what, budget, who, would, ve,...",yeah blow 3x what budget who would ve think gi...
2,"Food was pretty great. A little dry, but I am ...",en-AU,Google,1.0,0.0,"[food, pretty, great, little, dry, but, sucker...",food pretty great little dry but sucker load d...
3,Firstly the staff seemed as if they did n't wa...,en-UK,Google,0.0,0.0,"[firstly, staff, they, want, cheese, ham, toas...",firstly staff they want cheese ham toasty orde...
4,We came for lunch and enjoyed the food we orde...,en-UK,Google,1.0,0.0,"[we, come, lunch, enjoy, food, we, order, hot,...",we come lunch enjoy food we order hot taste fr...


Setup for Transformer model

In [ ]:
#Change Sentiment and Sarcasm to integer
train_df['Sentiment'] = train_df['Sentiment'].map({0.0: 0, 1.0: 1})
train_df['Sarcasm'] = train_df['Sarcasm'].map({0.0: 0, 1.0: 1})
val_df['Sentiment'] = val_df['Sentiment'].map({0.0: 0, 1.0: 1})
val_df['Sarcasm'] = val_df['Sarcasm'].map({0.0: 0, 1.0: 1})
test_df['Sentiment'] = test_df['Sentiment'].map({0.0: 0, 1.0: 1})
test_df['Sarcasm'] = test_df['Sarcasm'].map({0.0: 0, 1.0: 1})

In [ ]:
train_df['Sentiment'].value_counts(), train_df['Sarcasm'].value_counts()

(Sentiment
 0    1907
 1    1840
 Name: count, dtype: int64,
 Sarcasm
 0    3223
 1     524
 Name: count, dtype: int64)

In [ ]:
train_ds_sent = Dataset.from_pandas(train_df[["text", "Sentiment"]])
val_ds_sent = Dataset.from_pandas(val_df[["text", "Sentiment"]])
test_ds_sent = Dataset.from_pandas(test_df[["text", "Sentiment"]])

train_ds_sarc = Dataset.from_pandas(train_df[["text", "Sarcasm"]])
val_ds_sarc = Dataset.from_pandas(val_df[["text", "Sarcasm"]])
test_ds_sarc = Dataset.from_pandas(test_df[["text", "Sarcasm"]])



In [ ]:
train_ds_sent

Dataset({
    features: ['text', 'Sentiment'],
    num_rows: 3747
})

In [ ]:
train_ds_sent = train_ds_sent.rename_column("Sentiment", "label")
val_ds_sent   = val_ds_sent.rename_column("Sentiment", "label")
test_ds_sent  = test_ds_sent.rename_column("Sentiment", "label")

train_ds_sarc = train_ds_sarc.rename_column("Sarcasm", "label")
val_ds_sarc   = val_ds_sarc.rename_column("Sarcasm", "label")
test_ds_sarc  = test_ds_sarc.rename_column("Sarcasm", "label")

In [ ]:
train_ds_sent

Dataset({
    features: ['text', 'label'],
    num_rows: 3747
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [ ]:
def tokenize(batch):
  return tokenizer(
      batch["text"],
      truncation=True,
      padding=False,
      max_length=256
  )

In [ ]:
train_ds_sent = train_ds_sent.map(tokenize, batched=True)
val_ds_sent   = val_ds_sent.map(tokenize, batched=True)
test_ds_sent  = test_ds_sent.map(tokenize, batched=True)

train_ds_sarc = train_ds_sarc.map(tokenize, batched=True)
val_ds_sarc   = val_ds_sarc.map(tokenize, batched=True)
test_ds_sarc  = test_ds_sarc.map(tokenize, batched=True)

Map:   0%|          | 0/3747 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

Map:   0%|          | 0/2183 [00:00<?, ? examples/s]

Map:   0%|          | 0/3747 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

Map:   0%|          | 0/2183 [00:00<?, ? examples/s]

## Compute metrics for transformer model

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    _, _, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    cls_report = classification_report(labels, preds, output_dict=True)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "precision": macro_precision,
        "recall": macro_recall,
        "classification_report": cls_report
    }

### Basic Sentiment Analysis with Roberta

In [ ]:
model_sent = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(train_df["Sentiment"].unique()),
    problem_type="single_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
      output_dir = "./sentiment_results",
      weight_decay = 0.001,
      num_train_epochs = 3,
      learning_rate = 2e-5,
      eval_strategy = "epoch",
      save_strategy = "epoch",
      per_device_train_batch_size = 8,
      per_device_eval_batch_size = 8,
      report_to="none"
)

In [ ]:
trainer_sent = Trainer(
    model=model_sent,
    args=training_args,
    train_dataset=train_ds_sent,
    eval_dataset=val_ds_sent,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer_sent.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision,Recall,Classification Report
1,No log,0.193212,0.939297,0.939175,0.939236,0.940530,0.938766,"{'0': {'precision': 0.9221556886227545, 'recall': 0.9625, 'f1-score': 0.9418960244648318, 'support': 160.0}, '1': {'precision': 0.958904109589041, 'recall': 0.9150326797385621, 'f1-score': 0.9364548494983278, 'support': 153.0}, 'accuracy': 0.939297124600639, 'macro avg': {'precision': 0.9405298991058978, 'recall': 0.938766339869281, 'f1-score': 0.9391754369815798, 'support': 313.0}, 'weighted avg': {'precision': 0.9401189742708115, 'recall': 0.939297124600639, 'f1-score': 0.9392362807911094, 'support': 313.0}}"
2,0.402142,0.333375,0.907348,0.907333,0.907307,0.908823,0.908088,"{'0': {'precision': 0.9395973154362416, 'recall': 0.875, 'f1-score': 0.9061488673139159, 'support': 160.0}, '1': {'precision': 0.8780487804878049, 'recall': 0.9411764705882353, 'f1-score': 0.9085173501577287, 'support': 153.0}, 'accuracy': 0.9073482428115016, 'macro avg': {'precision': 0.9088230479620232, 'recall': 0.9080882352941176, 'f1-score': 0.9073331087358223, 'support': 313.0}, 'weighted avg': {'precision': 0.9095112903656001, 'recall': 0.9073482428115016, 'f1-score': 0.9073066241033836, 'support': 313.0}}"
3,0.252822,0.308792,0.932907,0.932863,0.932902,0.932944,0.932802,"{'0': {'precision': 0.9316770186335404, 'recall': 0.9375, 'f1-score': 0.9345794392523364, 'support': 160.0}, '1': {'precision': 0.9342105263157895, 'recall': 0.9281045751633987, 'f1-score': 0.9311475409836065, 'support': 153.0}, 'accuracy': 0.9329073482428115, 'macro avg': {'precision': 0.932943772474665, 'recall': 0.9328022875816994, 'f1-score': 0.9328634901179715, 'support': 313.0}, 'weighted avg': {'precision': 0.9329154425165567, 'recall': 0.9329073482428115, 'f1-score': 0.9329018659772066, 'support': 313.0}}"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1407, training_loss=0.2832009505954463, metrics={'train_runtime': 91.752, 'train_samples_per_second': 122.515, 'train_steps_per_second': 15.335, 'total_flos': 879828980238000.0, 'train_loss': 0.2832009505954463, 'epoch': 3.0})

In [ ]:
pred_output = trainer_sent.predict(test_ds_sent)
sentiment_metrics = compute_metrics((pred_output.predictions, pred_output.label_ids))
print(sentiment_metrics)

{'accuracy': 0.9056344480073294, 'macro_f1': 0.905617791702556, 'weighted_f1': 0.9056470838247436, 'precision': 0.9056011616975558, 'recall': 0.9058184865988871, 'classification_report': {'0': {'precision': 0.9159817351598174, 'recall': 0.8979409131602507, 'f1-score': 0.906871609403255, 'support': 1117.0}, '1': {'precision': 0.8952205882352942, 'recall': 0.9136960600375235, 'f1-score': 0.904363974001857, 'support': 1066.0}, 'accuracy': 0.9056344480073294, 'macro avg': {'precision': 0.9056011616975558, 'recall': 0.9058184865988871, 'f1-score': 0.905617791702556, 'support': 2183.0}, 'weighted avg': {'precision': 0.9058436762401921, 'recall': 0.9056344480073294, 'f1-score': 0.9056470838247436, 'support': 2183.0}}}


### Basic Sarcasm Model with Roberta

In [ ]:
model_sarcasm = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(train_df["Sarcasm"].unique()),
    problem_type="single_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
      output_dir = "./sarcasm_results",
      weight_decay = 0.001,
      num_train_epochs = 3,
      learning_rate = 2e-5,
      eval_strategy = "epoch",
      save_strategy = "epoch",
      per_device_train_batch_size = 8,
      per_device_eval_batch_size = 8,
      report_to="none"
)

In [ ]:
trainer_sarcasm = Trainer(
    model=model_sarcasm,
    args=training_args,
    train_dataset=train_ds_sarc,
    eval_dataset=val_ds_sarc,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)
trainer_sarcasm.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision,Recall,Classification Report
1,No log,0.431812,0.859425,0.462199,0.794451,0.429712,0.500000,"{'0': {'precision': 0.8594249201277955, 'recall': 1.0, 'f1-score': 0.9243986254295533, 'support': 269.0}, '1': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 44.0}, 'accuracy': 0.8594249201277955, 'macro avg': {'precision': 0.42971246006389774, 'recall': 0.5, 'f1-score': 0.46219931271477666, 'support': 313.0}, 'weighted avg': {'precision': 0.7386111933366677, 'recall': 0.8594249201277955, 'f1-score': 0.7944512148260379, 'support': 313.0}}"
2,0.381590,0.389027,0.849840,0.626672,0.834163,0.666278,0.608483,"{'0': {'precision': 0.8881118881118881, 'recall': 0.9442379182156134, 'f1-score': 0.9153153153153153, 'support': 269.0}, '1': {'precision': 0.4444444444444444, 'recall': 0.2727272727272727, 'f1-score': 0.3380281690140845, 'support': 44.0}, 'accuracy': 0.8498402555910544, 'macro avg': {'precision': 0.6662781662781663, 'recall': 0.608482595471443, 'f1-score': 0.6266717421646999, 'support': 313.0}, 'weighted avg': {'precision': 0.8257433017816405, 'recall': 0.8498402555910544, 'f1-score': 0.8341631286148228, 'support': 313.0}}"
3,0.303202,0.529950,0.856230,0.633518,0.838888,0.684444,0.612200,"{'0': {'precision': 0.8888888888888888, 'recall': 0.9516728624535316, 'f1-score': 0.9192100538599641, 'support': 269.0}, '1': {'precision': 0.48, 'recall': 0.2727272727272727, 'f1-score': 0.34782608695652173, 'support': 44.0}, 'accuracy': 0.8562300319488818, 'macro avg': {'precision': 0.6844444444444444, 'recall': 0.6122000675904021, 'f1-score': 0.6335180704082429, 'support': 313.0}, 'weighted avg': {'precision': 0.8314093006744764, 'recall': 0.8562300319488818, 'f1-score': 0.8388877070748156, 'support': 313.0}}"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1407, training_loss=0.32220442174763736, metrics={'train_runtime': 91.7254, 'train_samples_per_second': 122.551, 'train_steps_per_second': 15.339, 'total_flos': 879828980238000.0, 'train_loss': 0.32220442174763736, 'epoch': 3.0})

In [ ]:
pred_output = trainer_sarcasm.predict(test_ds_sarc)
sarcasm_metrics = compute_metrics((pred_output.predictions, pred_output.label_ids))
print(sarcasm_metrics)

{'accuracy': 0.8735684837379752, 'macro_f1': 0.6827636317684969, 'weighted_f1': 0.860044217798024, 'precision': 0.7396396396396396, 'recall': 0.6546430629026345, 'classification_report': {'0': {'precision': 0.9009009009009009, 'recall': 0.9584664536741214, 'f1-score': 0.9287925696594427, 'support': 1878.0}, '1': {'precision': 0.5783783783783784, 'recall': 0.35081967213114756, 'f1-score': 0.43673469387755104, 'support': 305.0}, 'accuracy': 0.8735684837379752, 'macro avg': {'precision': 0.7396396396396396, 'recall': 0.6546430629026345, 'f1-score': 0.6827636317684969, 'support': 2183.0}, 'weighted avg': {'precision': 0.8558393482809424, 'recall': 0.8735684837379752, 'f1-score': 0.860044217798024, 'support': 2183.0}}}


### Fine tuning the model

In [ ]:

def find_best_config_from_grid(candidate_config, train_dataset, val_dataset):
    results = []
    for config in candidate_config:
        print(config)
        print(f"Running: lr={config['learning_rate']}, bs={config['batch_size']}, epochs={config['num_train_epochs']}, wd={config['weight_decay']}")

        set_seed(42)

        model = AutoModelForSequenceClassification.from_pretrained(
            "roberta-base",
            num_labels=2
        )


        training_args = TrainingArguments(
            output_dir=f"./results/lr_{config['learning_rate']}_bs_{config['batch_size']}_ep_{config['num_train_epochs']}",
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            fp16=torch.cuda.is_available(),
            dataloader_pin_memory=True,
            learning_rate=config['learning_rate'],
            per_device_train_batch_size=config['batch_size'],
            per_device_eval_batch_size=config['batch_size'],
            num_train_epochs=config['num_train_epochs'],
            weight_decay=config['weight_decay'],
            seed=42,
            warmup_ratio=0.1,
            lr_scheduler_type="linear"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
        )

        trainer.train()

        pred_output = trainer.predict(val_dataset)
        eval_result = compute_metrics((pred_output.predictions, pred_output.label_ids))


        results.append({
            "learning_rate": config['learning_rate'],
            "batch_size": config['batch_size'],
            "num_train_epochs": config['num_train_epochs'],
            "weight_decay": config['weight_decay'],
            "accuracy": eval_result["accuracy"],
            "macro_f1": eval_result["macro_f1"],
            "weighted_f1": eval_result["weighted_f1"],
            "precision": eval_result["precision"],
            "recall": eval_result["recall"]
        })

        del model
        torch.cuda.empty_cache()


    return results


In [ ]:
candidate_configs = [
    {'learning_rate': 1e-5, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.01},
    {'learning_rate': 1e-5, 'batch_size': 8, 'num_train_epochs': 3, 'weight_decay': 0.0},
    {'learning_rate': 1e-5, 'batch_size': 8, 'num_train_epochs': 3, 'weight_decay': 0.01},
    {'learning_rate': 2e-5, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.0},
    {'learning_rate': 2e-5, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.01},
    {'learning_rate': 2e-5, 'batch_size': 8, 'num_train_epochs': 3, 'weight_decay': 0.0},
    {'learning_rate': 2e-5, 'batch_size': 8, 'num_train_epochs': 3, 'weight_decay': 0.01},
    {'learning_rate': 3e-5, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.0},
    {'learning_rate': 3e-5, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.01},
    {'learning_rate': 3e-5, 'batch_size': 8, 'num_train_epochs': 3, 'weight_decay': 0.0},
    {'learning_rate': 3e-5, 'batch_size': 8, 'num_train_epochs': 3, 'weight_decay': 0.01},

    {'learning_rate': 1e-5, 'batch_size': 16, 'num_train_epochs': 2, 'weight_decay': 0.01},
    {'learning_rate': 1e-5, 'batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.0},
    {'learning_rate': 1e-5, 'batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.01},
    {'learning_rate': 2e-5, 'batch_size': 16, 'num_train_epochs': 2, 'weight_decay': 0.0},
    {'learning_rate': 2e-5, 'batch_size': 16, 'num_train_epochs': 2, 'weight_decay': 0.01},
    {'learning_rate': 2e-5, 'batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.0},
    {'learning_rate': 2e-5, 'batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.01},
    {'learning_rate': 3e-5, 'batch_size': 16, 'num_train_epochs': 2, 'weight_decay': 0.0},
    {'learning_rate': 3e-5, 'batch_size': 16, 'num_train_epochs': 2, 'weight_decay': 0.01},
    {'learning_rate': 3e-5, 'batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.0},
    {'learning_rate': 3e-5, 'batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.01},

]

In [ ]:
#Sentiment tuning
sentiment_results = find_best_config_from_grid(candidate_configs, train_ds_sent, val_ds_sent)

{'learning_rate': 1e-05, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.01}
Running: lr=1e-05, bs=8, epochs=2, wd=0.01


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision,Recall,Classification Report
1,No log,0.208407,0.929712,0.929677,0.929712,0.929677,0.929677,"{'0': {'precision': 0.93125, 'recall': 0.93125, 'f1-score': 0.93125, 'support': 160.0}, '1': {'precision': 0.9281045751633987, 'recall': 0.9281045751633987, 'f1-score': 0.9281045751633987, 'support': 153.0}, 'accuracy': 0.9297124600638977, 'macro avg': {'precision': 0.9296772875816994, 'recall': 0.9296772875816994, 'f1-score': 0.9296772875816994, 'support': 313.0}, 'weighted avg': {'precision': 0.9297124600638977, 'recall': 0.9297124600638977, 'f1-score': 0.9297124600638977, 'support': 313.0}}"
2,0.430946,0.245393,0.939297,0.939257,0.939292,0.939339,0.939195,"{'0': {'precision': 0.937888198757764, 'recall': 0.94375, 'f1-score': 0.940809968847352, 'support': 160.0}, '1': {'precision': 0.9407894736842105, 'recall': 0.934640522875817, 'f1-score': 0.9377049180327869, 'support': 153.0}, 'accuracy': 0.939297124600639, 'macro avg': {'precision': 0.9393388362209872, 'recall': 0.9391952614379084, 'f1-score': 0.9392574434400695, 'support': 313.0}, 'weighted avg': {'precision': 0.9393063938496052, 'recall': 0.939297124600639, 'f1-score': 0.9392921644555677, 'support': 313.0}}"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

{'learning_rate': 3e-05, 'batch_size': 8, 'num_train_epochs': 2, 'weight_decay': 0.0}
Running: lr=3e-05, bs=8, epochs=2, wd=0.0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision,Recall,Classification Report
1,No log,0.328927,0.878594,0.878145,0.877980,0.888226,0.880392,"{'0': {'precision': 0.9552238805970149, 'recall': 0.8, 'f1-score': 0.8707482993197279, 'support': 160.0}, '1': {'precision': 0.8212290502793296, 'recall': 0.9607843137254902, 'f1-score': 0.8855421686746988, 'support': 153.0}, 'accuracy': 0.8785942492012779, 'macro avg': {'precision': 0.8882264654381722, 'recall': 0.8803921568627451, 'f1-score': 0.8781452339972133, 'support': 313.0}, 'weighted avg': {'precision': 0.8897248101861335, 'recall': 0.8785942492012779, 'f1-score': 0.8779798073430842, 'support': 313.0}}"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Sarcasm tuning
sarcasm_results = find_best_config_from_grid(candidate_configs, train_ds_sarc, val_ds_sarc)

In [ ]:
sent_df = pd.DataFrame(sentiment_results)

sent_df_sorted = sent_df.sort_values(by="macro_f1", ascending=False)

print(sent_df_sorted.head())

## Running the best sentiment model

In [ ]:
best_config_sent = sent_df_sorted.iloc[0].to_dict()
print("Best config:", best_config_sent)

In [ ]:
full_train_dataset_sent = concatenate_datasets([train_ds_sent, val_ds_sent])

In [ ]:
seeds = [42, 100]

all_results = {}

for seed in seeds:
    print(f"\nRunning with seed: {seed}")

    set_seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"./sentiment_final_model_seed_{seed}",
        eval_strategy="no",
        learning_rate=best_config_sent["learning_rate"],
        per_device_train_batch_size=int(best_config_sent["batch_size"]),
        num_train_epochs=int(best_config_sent["num_train_epochs"]),
        weight_decay=best_config_sent["weight_decay"],
        fp16=True,
        seed=seed
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_train_dataset_sent,
        processing_class=tokenizer
    )

    trainer.train()

    pred_output = trainer.predict(test_ds_sent)
    eval_result = compute_metrics((pred_output.predictions, pred_output.label_ids))
    all_results[seed] = eval_result

In [ ]:
all_results

In [ ]:
trainer.save_model("drive/MyDrive/nlp/best_sentiment_model")
tokenizer.save_pretrained("drive/MyDrive/nlp/best_sentiment_model")

## Running the best sarcasm model

In [ ]:
sarc_df = pd.DataFrame(sarcasm_results)

sarc_df_sorted = sarc_df.sort_values(by="macro_f1", ascending=False)

print(sarc_df_sorted.head())

In [ ]:
best_config_sarc = sarc_df_sorted.iloc[0].to_dict()
print("Best config:", best_config_sarc)

In [ ]:
full_train_dataset_sarc = concatenate_datasets([train_ds_sarc, val_ds_sarc])

In [ ]:
from transformers import set_seed

seeds = [42, 100]

all_results_sarc = {}

for seed in seeds:
    print(f"\nRunning sarcasm model with seed: {seed}")

    set_seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"./sarcasm_final_model_seed_{seed}",
        eval_strategy="no",
        learning_rate=best_config_sarc["learning_rate"],
        per_device_train_batch_size=int(best_config_sarc["batch_size"]),
        num_train_epochs=int(best_config_sarc["num_train_epochs"]),
        weight_decay=best_config_sarc["weight_decay"],
        fp16=True,
        seed=seed
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_train_dataset_sarc,
    )

    trainer.train()

    pred_output = trainer.predict(test_ds_sarc)
    eval_result = compute_metrics((pred_output.predictions, pred_output.label_ids))
    all_results_sarc[seed] = eval_result


In [ ]:
all_results_sarc

In [ ]:
trainer.save_model("drive/MyDrive/nlp/best_sarcasm_model")
tokenizer.save_pretrained("drive/MyDrive/nlp/best_sarcasm_model")